# 05 · Processes and Collaboration

> **Kernel:** `Phase 5 · CrewAI` · **Needs:** Ollama. This notebook compares
> costs, so it runs several crews; expect a few minutes.

A crew's **process** decides how its tasks run: one after another, several at
once, or under a **manager** that plans and delegates. Agents can also
**delegate** to each other. These choices change more than the order of work;
they change **cost and predictability**, sometimes by an order of magnitude.
This notebook measures that.

**You will learn**
- The **sequential** process, and how context flows between tasks
- **Parallel tasks** with `async_execution`, and how to join their results
- The **hierarchical** process: a manager that plans and delegates, and what it costs
- **Delegation** between agents (`allow_delegation`)
- How to choose, based on measured calls and time

**Prerequisites:** notebooks 01–04.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')                       # hide noisy library warnings in the notebook
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 03_CREWAI folder
sys.path.insert(0, str(SECTION))
from common import get_llm, quiet_console              # import common BEFORE crewai (telemetry off)
quiet_console()                                         # hide CrewAI's decorative console panels

import time
from crewai import Agent, Task, Crew, Process


async def measure(label, crew, **kwargs):
    # Run a crew and report wall time and model calls. Agents get fresh LLMs, so counts are per run.
    t0 = time.perf_counter()
    out = await crew.kickoff_async(**kwargs)
    u = out.token_usage
    print(f'{label:<28} {time.perf_counter() - t0:5.1f}s   {u.successful_requests:>3} model calls   {u.total_tokens:>6} tokens')
    return out

## 1 · Why processes matter

Every crew answers the same question: **who decides which agent does what, and
in which order?**

| Process | Who decides | Order |
|---|---|---|
| **sequential** | you: tasks run in the order you listed, each with an assigned agent | fixed |
| sequential + `async_execution` | you: marked tasks run concurrently | fixed, partly parallel |
| **hierarchical** | a **manager** agent: plans, assigns work to agents, reviews results | decided at runtime |

The more the framework decides, the less *you* have to specify, and the more
model calls it spends deciding. Every measurement below uses a **fresh LLM per
agent** so the counts belong to that run (notebook 02 explains why that matters).

## 2 · Sequential: fixed order, automatic context

The default. Each task runs after the previous one and receives its output as
context. A three-step pipeline: research → draft → edit.

In [ ]:
def pipeline():
    researcher = Agent(role='Researcher', goal='Find key facts', backstory='Precise.', llm=get_llm())
    writer = Agent(role='Writer', goal='Draft short texts', backstory='Clear and brief.', llm=get_llm())
    editor = Agent(role='Editor', goal='Polish texts', backstory='Strict about clarity.', llm=get_llm())
    research = Task(description='List 3 facts about {topic}.', expected_output='3 bullets', agent=researcher)
    draft = Task(description='Write a 3-sentence paragraph from the facts.', expected_output='3 sentences', agent=writer)
    edit = Task(description='Tighten the paragraph to at most 50 words.', expected_output='One paragraph, max 50 words', agent=editor)
    return Crew(agents=[researcher, writer, editor], tasks=[research, draft, edit], process=Process.sequential)

out = await measure('sequential (3 tasks)', pipeline(), inputs={'topic': 'sidewalk delivery robots'})
print('\n' + out.raw)

Three tasks, roughly one model call each: sequential crews are **cheap and
predictable**. Their cost is time, since nothing overlaps.

## 3 · Parallel tasks with `async_execution`

Independent tasks don't need to wait for each other. Mark them
`async_execution=True` and they run concurrently. A later task lists them in its
`context` to **join** their results:

```
 legal analyst ──┐
 tech analyst  ──┼──(context)──► judge: go / no-go
 market analyst ─┘
   async_execution=True
```

In [ ]:
def panel(parallel: bool):
    analysts, tasks = [], []
    for area in ('legal', 'technical', 'market'):
        a = Agent(role=f'{area} analyst', goal=f'Assess {area} risk', backstory='One sentence, no preamble.', llm=get_llm())
        analysts.append(a)
        tasks.append(Task(description=f'In one sentence: the biggest {area} risk of {{idea}}.',
                          expected_output='One sentence', agent=a, async_execution=parallel))
    judge = Agent(role='Judge', goal='Decide', backstory='Decisive.', llm=get_llm())
    verdict = Task(description='Given the three risk notes, decide go or no-go in one sentence.',
                   expected_output='One sentence', agent=judge, context=tasks)     # ← the join
    return Crew(agents=analysts + [judge], tasks=tasks + [verdict])

IDEA = {'idea': 'robots that carry groceries up apartment stairs'}
await measure('3 analysts, one by one', panel(parallel=False), inputs=IDEA)
out = await measure('3 analysts, in parallel', panel(parallel=True), inputs=IDEA)
print('\nverdict:', out.raw)

Same model calls, less wall-clock time. How much less depends on your hardware:
a single local GPU partly serialises concurrent requests, while a cloud API
runs them truly in parallel.

**Rules for async tasks:** the tasks that consume them must list them in
`context` (that's how CrewAI knows to wait for them), and a crew shouldn't end
on a bare async task. Put a joining task last.

## 4 · Hierarchical: a manager plans and delegates

In a hierarchical crew you give **tasks without assigned agents**. A **manager**
(an LLM you provide with `manager_llm`, or your own `manager_agent`) reads each
task, decides which agent should do it, delegates, checks the result, and may
send work back. It sounds ideal. Let's measure one simple task done both ways:

In [ ]:
TASK = 'Write a 2-sentence explanation of why delivery robots need curb detection.'

def direct():
    writer = Agent(role='Writer', goal='Write clear explanations', backstory='Concise.', llm=get_llm())
    return Crew(agents=[writer], tasks=[Task(description=TASK, expected_output='2 sentences', agent=writer)])

def managed():
    researcher = Agent(role='Researcher', goal='Find facts', backstory='Thorough.', llm=get_llm())
    writer = Agent(role='Writer', goal='Write clear explanations', backstory='Concise.', llm=get_llm())
    return Crew(agents=[researcher, writer], tasks=[Task(description=TASK, expected_output='2 sentences')],
                process=Process.hierarchical, manager_llm=get_llm())

await measure('sequential, 1 agent', direct())
out = await measure('hierarchical, manager + 2', managed())
print('\n' + out.raw)

The hierarchical crew produced a similar two sentences, at a much higher cost:
the manager read the task, delegated it (with long delegation prompts listing
its coworkers), then reviewed and returned the result. In our runs the direct
crew used **1 call and ~120 tokens**, and the managed crew **3 calls and ~2,400
tokens: about 20× the tokens and several times the time**, for the same answer.

**A measuring lesson we learned the hard way.** Our first measurement of this
comparison reported **36 model calls** for the hierarchical crew. That number
was wrong: the test script reused one LLM object across several crews, so
`token_usage` was the running total from notebook 02, not this run's cost. With a
fresh LLM per agent (as `measure` uses here) the real figure is 3 calls. Before
you draw conclusions about cost, make sure you're measuring one run.

That overhead can pay off when tasks are genuinely open-ended and you *can't*
decide in advance who should do what. For work you can assign yourself, it's
pure cost, and the manager's choices add unpredictability. Measure before you
choose it.

## 5 · Delegation between agents

`allow_delegation=True` (default **False**) gives an agent two extra tools: *delegate
work to a coworker* and *ask a coworker a question*. The agent decides at runtime
whether to use them. The hierarchical manager uses exactly these tools.

In [ ]:
def with_delegation():
    lead = Agent(role='Team lead', goal='Deliver accurate answers',
                 backstory='You delegate calculations to the analyst instead of doing them yourself.',
                 llm=get_llm(), allow_delegation=True)
    analyst = Agent(role='Analyst', goal='Do calculations exactly', backstory='Careful with numbers.', llm=get_llm())
    task = Task(description='We ship 180 robots and 48 are delayed. What percentage is delayed? Delegate the calculation.',
                expected_output='The percentage, one sentence', agent=lead)
    return Crew(agents=[lead, analyst], tasks=[task])

out = await measure('lead with delegation', with_delegation())
print('\n' + out.raw)

Delegation lets an agent pull in a specialist without you wiring it, at the cost
of extra model calls and less control over who does what. Keep it off (the
default) unless you want that runtime flexibility.

**Planning.** `Crew(planning=True)` adds a planning step: before running, a
planner LLM writes a step-by-step plan for every task and adds it to each task's
prompt. It can improve multi-step work, and costs extra calls up front.

## 6 · How to choose

| Situation | Choose |
|---|---|
| You know the steps and who does each | sequential |
| Some steps are independent | sequential + `async_execution` on those, a joining task after |
| You can't know in advance which specialist should handle what | hierarchical (measure the cost first) |
| One agent occasionally needs a specialist's help | `allow_delegation=True` on that agent |
| Steps need branching, loops, state or human pauses | a **Flow** (notebook 06) calling crews |

## 7 · Common mistakes

- **Choosing hierarchical because it sounds smarter.** It multiplied the tokens ~20×
  for a simple task here. Measure against sequential, with fresh LLM objects.
- **Async tasks without a join.** Consumers must list them in `context`.
- **Delegation everywhere.** Agents ping-pong work and the cost climbs.
- **Measuring cost across shared LLM objects.** Counts accumulate (notebook 02).
- **Forcing control flow into a crew.** Branching and loops belong in Flows.

## 8 · Check yourself

<details><summary><b>1.</b> In a sequential crew, what does each task receive by default?</summary>

The output of the previous task, as context. Use `context=[...]` to choose different or additional inputs.
</details>

<details><summary><b>2.</b> How do you run three independent tasks at once and combine their results?</summary>

Mark them `async_execution=True`, then add a later task whose `context` lists all three. It waits for them and receives their outputs.
</details>

<details><summary><b>3.</b> What does a hierarchical manager do, and why did it cost so much more here?</summary>

It decides which agent handles each task, delegates, reviews results and may iterate. Each of those steps is a model call with long coordination prompts. For a task one agent could do directly, that coordination is pure overhead (here ~20× the tokens).
</details>

<details><summary><b>4.</b> What does <code>allow_delegation=True</code> give an agent?</summary>

Two tools: delegate work to a coworker, and ask a coworker a question. The agent decides at runtime whether to use them.
</details>

## Takeaway

Sequential crews are cheap and predictable, and `async_execution` plus a joining
task adds parallelism for independent work. Hierarchical crews and delegation
let agents decide who does what, at a cost we measured at about **20× the tokens**
for a simple task. Choose them only when that runtime flexibility is
worth it, and move real control flow into Flows.

Next → `06_flows.ipynb`: explicit, stateful control flow.